# LangChain Standard Output Tracers Reference

Developer-facing statements defined in `langchain_core.tracers.stdout`.

# `MILLISECONDS_IN_SECOND`

Number of milliseconds in one second, used when formatting sub-second run durations.

```python
MILLISECONDS_IN_SECOND = 1000
```
# `try_json_stringify`

Converts an object to an indented JSON string when possible.

```python
try_json_stringify(
    obj: Any, # Object to convert to JSON
    fallback: str, # Value returned when JSON serialization fails
) -> str # JSON string or the supplied fallback
```

The JSON output uses an indentation of two spaces and preserves non-ASCII characters. Any exception raised during serialization causes the function to return `fallback`.

---

# `elapsed`

Formats the elapsed duration of an object having `start_time` and `end_time` attributes.

```python
elapsed(
    run: Any, # Object whose start_time and end_time are subtracted
) -> str # Duration formatted in milliseconds or seconds
```

Durations below one second are rounded to the nearest whole millisecond and suffixed with `"ms"`. Durations of at least one second are formatted with two decimal places and suffixed with `"s"`.

# `FunctionCallbackHandler: BaseTracer`

Tracer that sends formatted trace messages to a callback accepting one string argument.

## Fields

```python
name: str = "function_callback_handler" # Name used to identify the tracer in logs
```

## Constructor

```python
FunctionCallbackHandler(
    function: Callable[[str], None], # Callback receiving each formatted trace message
    **kwargs: Any, # Arguments forwarded to BaseTracer
) -> None
```

The callback is stored on the instance as `function_callback`.

## Methods

### `get_parents`

Returns the available ancestor runs of a run.

```python
get_parents(
    self,
    run: Run, # Run whose ancestors should be retrieved
) -> list[Run] # Ancestors ordered from immediate parent toward the root
```

The method follows `parent_run_id` values through the active `run_map`. Traversal stops when a parent ID is absent or its run is not available.

### `get_breadcrumbs`

Builds a breadcrumb string for a run and its available ancestors.

```python
get_breadcrumbs(
    self,
    run: Run, # Run whose breadcrumb path should be created
) -> str # Root-to-current path formatted as run-type and name pairs
```

Each path component uses the form `"<run_type>:<name>"`, and components are joined with `" > "`.

## Behaviour

The handler emits color-formatted messages for chain, LLM, and tool start, completion, and error events.

Chain messages include breadcrumbs and JSON-formatted inputs, outputs, or errors. Completion and error messages include the formatted elapsed duration.

LLM start messages strip surrounding whitespace from each prompt when the run inputs contain `"prompts"`. LLM completion and error messages include breadcrumbs, elapsed duration, and JSON-formatted response or error data.

Tool start messages read and strip `run.inputs["input"]`. Tool completion messages are emitted only when `run.outputs` is truthy and use the stripped string representation of `run.outputs["output"]`. Tool error messages include breadcrumbs, elapsed duration, and the run error.

Completed root runs are not persisted by this tracer.

In [ ]:
from langchain_core.runnables import RunnableLambda # Import the real LangChain runnable
from langchain_core.tracers.stdout import FunctionCallbackHandler # Import the function-based tracer


trace_messages: list[str] = [] # Store every formatted trace message


def receive_trace(message: str) -> None: # Define the function called for each trace message
    trace_messages.append(message) # Save the trace message
    print(message) # Display the trace message in Jupyter


def add_two(number: int) -> int: # Define the first pipeline operation
    return number + 2 # Add two to the input


def multiply_by_three(number: int) -> int: # Define the second pipeline operation
    return number * 3 # Multiply the input by three


add_step = RunnableLambda(add_two).with_config(run_name="add_two") # Create the first named runnable
multiply_step = RunnableLambda(multiply_by_three).with_config(run_name="multiply_by_three") # Create the second named runnable

pipeline = add_step | multiply_step # Combine both operations into one pipeline

handler = FunctionCallbackHandler(receive_trace) # Send formatted traces to receive_trace

result = pipeline.invoke( # Run the real pipeline
    5, # Provide the input value
    config={ # Configure the traced execution
        "callbacks": [handler], # Attach the function callback handler
        "run_name": "math_pipeline", # Name the root run
        "tags": ["math", "demo"], # Add run tags
        "metadata": {"source": "jupyter"}, # Add run metadata
    },
) # Finish invoking the pipeline

print("\nFinal result:", result) # Display the final pipeline result
print("Trace messages received:", len(trace_messages)) # Display the number of callback messages
print("Handler name:", handler.name) # Display the handler name

# `ConsoleCallbackHandler: FunctionCallbackHandler`

Tracer that prints formatted trace messages to standard output.

## Fields

```python
name: str = "console_callback_handler" # Name used to identify the tracer in logs
```

## Constructor

```python
ConsoleCallbackHandler(
    **kwargs: Any, # Arguments forwarded through FunctionCallbackHandler to BaseTracer
) -> None
```

The constructor configures `print` as the callback function.

In [ ]:
from langchain_core.runnables import RunnableLambda # Import the real LangChain runnable
from langchain_core.tracers.stdout import ConsoleCallbackHandler # Import the console tracer


def add_two(number: int) -> int: # Define the first operation
    return number + 2 # Add two to the input


def multiply_by_three(number: int) -> int: # Define the second operation
    return number * 3 # Multiply the input by three


add_step = RunnableLambda(add_two).with_config(run_name="add_two") # Create the first named runnable
multiply_step = RunnableLambda(multiply_by_three).with_config(run_name="multiply_by_three") # Create the second named runnable

pipeline = add_step | multiply_step # Combine both operations into one pipeline
console_handler = ConsoleCallbackHandler() # Create the console callback handler

result = pipeline.invoke( # Run the pipeline and print trace messages
    5, # Provide the input
    config={ # Configure this execution
        "callbacks": [console_handler], # Attach the console tracer
        "run_name": "math_pipeline", # Name the root run
        "tags": ["math", "demo"], # Add trace tags
        "metadata": {"source": "jupyter"}, # Add trace metadata
    },
) # Finish invoking the pipeline

print("\nFinal result:", result) # Display the final result